This notebook is part of a video tutorial! See [here](https://youtu.be/9ZjEUKQhNw4) if you'd like to listen to an explanation of the code.

---

In [ ]:
# Load libraries
import numpy as np
import pandas as pd
import torch

In [ ]:
# Load test data
df = pd.read_csv("./framingham.csv")    # this is a dataset of patient records, with the goal of predicting whether or not they will have a heart attack in the next 10 years
df.head()

In [ ]:
# Prep data to train a logistic regression model on

# Drop rows with any null values
df_clean = df.dropna()

# Separate features (X) and target (y)
X = df_clean.drop('TenYearCHD', axis=1).values
y = df_clean['TenYearCHD'].values

In [ ]:
X = torch.from_numpy(X).float()
y = torch.from_numpy(y).float()
print(X.shape)
print(y.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Create and fit scaler on the data
scaler = StandardScaler()
X_normalized = torch.from_numpy(scaler.fit_transform(X.numpy())).float()

# Replace X with normalized version
X = X_normalized
print(X.shape)
print(X)

# Level 1: Use For Vector Math
This is just using Pytorch as NumPy because we lack imagination.

![meme 1](https://mat3e.github.io/brains/img/0.jpg)

In [ ]:
# BAD (numerically unstable AKA get very large outputs if x gets too large or too small)
def sigmoid(x):
    return  1 / (1 + torch.exp(-x))

In [ ]:
def sigmoid(x):
    # Applies the most stable formula for each el
    return torch.where(
        x >= 0,
        # e^(-x) won't be large for x > 0
        1 / (1 + torch.exp(-x)),
        # Multiplied above equation by (e^x)/(e^x) since
        # e^(-x) could get large for x < 0
        torch.exp(x) / (1 + torch.exp(x))
    )

# gradient of sigmoid(x) with respect to x
def sigmoid_prime(x):
    return sigmoid(x) * (1 - sigmoid(x))

In [ ]:
# Logistic regression
def feedforward(X,theta):
    z = X @ theta                 #Weighted input
    a = sigmoid(z)                #Activation
    return a

In [ ]:
# Not the best cost function for logistic regression, but simple
def cost_function(pred,y):
    m = y.shape[0]
    
    diff_squared = (pred - y) ** 2
    avg_per_example = torch.sum(diff_squared) / m
    return avg_per_example

In [ ]:
def cost_function(y_pred, y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1.0 - epsilon)
    
    # The exact math formula
    loss = -(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred))
    return torch.mean(loss)

In [ ]:
def gradient_vectorised(pred,X,y):
    m = y.shape[0]
    
    cost_by_pred = 2 * (pred - y)                           #This returns an m x 1 vector
    sigmoid_out_by_sigmoid_in = sigmoid_prime(X)            #This returns a m x n vector
    X_sum = torch.sum(X, axis=0)                               #X was m x n. Now it's 1 x n
    
    gradient = cost_by_pred @ sigmoid_out_by_sigmoid_in * X_sum
    return gradient / m

In [ ]:
def gradient_vectorised(pred,X,y):
    m = y.shape[0]
    
    error = pred - y
    gradient = (X.T @ error) / m

    return gradient

In [ ]:
def gradient_descent(X,y,theta,learning_rate,num_iters):
    cost_history = []
    for i in range(num_iters):
        pred = feedforward(X,theta)
        cost = cost_function(pred,y)                   #This is just to monitor cost over iterations
        cost_history.append(float(cost))
        
        gradient = gradient_vectorised(pred,X,y)          #This is the backpropagation part
        theta = theta - learning_rate * gradient
    
    return theta, cost_history
    
        
learning_rate = 0.01
num_iters = 100
theta = torch.randn(X.shape[-1])
theta, cost_history = gradient_descent(X,y,theta,learning_rate,num_iters)
print(cost_history)

# Level 2: I HATE BACKPROPAGATION
Let's use Pytorch to skip the annoying gradient calculations

![meme 2](https://mat3e.github.io/brains/img/1.jpg)

In [ ]:
def gradient_descent(X,y,theta,learning_rate,num_iters):
    # Tell Pytorch to track computations for this tensor
    # so we can compute gradients later
    theta = theta.clone().detach().requires_grad_(True)
    cost_history = []
    
    for i in range(num_iters):
        # Pytorch is watching these computations
        pred = feedforward(X, theta)
        cost = cost_function(pred,y)                   
        cost_history.append(float(cost))

        # Automatically computing the gradient with chain rule
        cost.backward()
        
        # Don't track these calculations
        with torch.no_grad():
            # Gradient descent param update if gradient is normal
            if (not torch.isnan(theta.grad).any()):
                theta -= learning_rate * theta.grad
                
        # Reset gradient calc for next iteration
        theta.grad.zero_()

    return theta.detach(), cost_history

In [ ]:
# Use while still debugging code.
# torch.autograd.set_detect_anomaly(True)
theta = torch.randn(X.shape[-1])

# Run gradient descent
learning_rate = 0.1
num_iters = 100
theta_with_pytorch, cost_history = gradient_descent(X,y,theta,learning_rate,num_iters)
print()
print(cost_history)

Yes, you're seeing correctly this isn't working very well. That's because the functions we're using aren't very numerically stable and oversimplified. Let's learn about more pytorch features that can get this to work better.

# Level 3: Add Some Variety!
Let's say we want to change our cost function. Or our prediction equation. Manually changing code is annoying. Luckily, Pytorch has some libraries to help us experiment!

The magic libraries are [torch.nn](https://pytorch.org/docs/stable/nn.html) and [torch.nn.functional](https://pytorch.org/docs/stable/nn.functional.html)

![meme 3](https://mat3e.github.io/brains/img/2.jpg)

In [ ]:
# torch.nn.functional has functions!
import torch.nn.functional as F

bce = F.binary_cross_entropy
cross_entropy = F.cross_entropy
sigmoid = torch.sigmoid
relu = F.relu
linear = F.linear
mse = F.mse_loss

In [ ]:
def feedforward(X, theta):
    return sigmoid(linear(X, theta))

pred = feedforward(X, theta_with_pytorch)
# Binary cross entropy is better than MSE for logistic regression
# With pytorch, we can just use it without changing lots of code
cost = bce(pred, y)

In [ ]:
def gradient_descent(X,y,theta,learning_rate,num_iters):
    theta = theta.clone().detach().requires_grad_(True)
    cost_history = []
    
    for i in range(num_iters):
        print(".", end="")

        pred = sigmoid(linear(X, theta))
        cost = bce(pred,y)                   
        cost_history.append(float(cost))

        cost.backward()

        with torch.no_grad():
            if (not torch.isnan(theta.grad).any()):
                theta -= learning_rate * theta.grad
        theta.grad.zero_()

    return theta.detach(), cost_history

# Run gradient descent
theta = torch.randn(X.shape[-1])
learning_rate = 0.1
num_iters = 100
theta_with_pytorch, cost_history = gradient_descent(X,y,theta,learning_rate,num_iters)
print()
print(cost_history)

# Level 4: Keeping Variables Organised
Just like Python has 'modules' where coding tools are stored, Pytorch has a **class** called `torch.nn.Module` that's built to store common tools used in neural networks.

One of the common 'tools' used in neural networks are its parameters! A Pytorch class called `torch.nn.Parameter` stores those.

![meme 4](https://mat3e.github.io/brains/img/3.jpg)

In [ ]:
# To use the code in torch.nn.Module class, we have to create a child class with our own code
class my_model(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = torch.nn.Parameter(torch.randn(X.shape[-1]))
        
    def forward(self, X):     # This forward method is a 'reserved' name with Pytorch! Careful!
        return sigmoid(linear(X, self.theta))

    def cost(self, pred, y):
        return bce(pred, y)

child_instance = my_model()
pred = child_instance(X)
cost = child_instance.cost(pred, y)

In [ ]:
def gradient_descent(X,y,num_iters,learning_rate):
    cost_history = []
    for i in range(num_iters):
        pred = child_instance(X)
        cost = child_instance.cost(pred, y)
        cost_history.append(float(cost))
        
        cost.backward()
        with torch.no_grad():
            for p in child_instance.parameters():        # Really helpful for multiple params
                p -= learning_rate * p.grad
        child_instance.zero_grad()                      # Really helpful for multiple params
    
    return cost_history

In [ ]:
learning_rate = 0.1
cost_history = gradient_descent(X,y,num_iters,learning_rate)
print(cost_history)

# Level 5: But could you just do everything for me?
First, we'll use `torch.nn.[Layer_Name]` as a class that creates its own parameters.

Then, we'll use `torch.nn.Sequential` as a class that stores all these classes.

![meme 5](https://mat3e.github.io/brains/img/4.jpg)

In [ ]:
import torch.nn as nn

lr_model = nn.Sequential(        # A predefined class that takes parameters from the layers we pass
    nn.Linear(X.shape[-1],1),       # A predefined class that makes its own parameters
    nn.Sigmoid(),
)

pred = lr_model(X)
# .squeeze() gets rid of extra dimensions. Ex: [1, 5] -> [5]
cost = torch.nn.functional.binary_cross_entropy(pred.squeeze(-1), y)
print(cost)

Also, never mind that whole gradient descent thing. Let's use `torch.optim` to take care of that.

In [ ]:
def gradient_desc(mod,X,y,learning_rate,num_iters):
    opt = torch.optim.SGD(mod.parameters(), lr=learning_rate)
    cost_history = []
    
    for i in range(num_iters):
        opt.zero_grad()            # YASSSS
        pred = mod(X)
        cost = torch.nn.functional.binary_cross_entropy(pred.squeeze(-1), y)
        cost_history.append(cost.item())
        
        cost.backward()            # Backpropagation
        opt.step()                 # YASSSS

    return cost_history

learning_rate = 0.1
cost_history = gradient_desc(lr_model,X,y,learning_rate,num_iters)
print(cost_history)

Here's another example showing a more complicated model because it's easy to create those now

In [ ]:
import torch.nn.functional as F
import torch.nn as nn

neural_net = nn.Sequential(
    nn.Linear(X.shape[-1],10),
    nn.ReLU(),
    nn.Linear(10,5),
    nn.ReLU(),
    nn.Linear(5,1),
    nn.Sigmoid()
)

def gradient_descent(mod,X,y,learning_rate,num_iters):
    opt = torch.optim.SGD(mod.parameters(), lr=learning_rate)
    cost_history = []
    
    for i in range(num_iters):
        opt.zero_grad()            # YASSSS
        pred = mod(X)
        cost = F.binary_cross_entropy(pred.squeeze(-1), y)
        cost_history.append(cost.item())
        
        cost.backward()            # Backpropagation
        opt.step()                 # YASSSS

    return cost_history

learning_rate = 0.01
cost_history = gradient_descent(neural_net, X,y,learning_rate,num_iters)
print(cost_history)

# Level 6: Simplify that Data Stuff Too
Some useful libraries for preprocessing are: `torch.utils.data`, `torchvision.transforms`, and `torchtext.data.utils`.

Some useful libraries for data are: `torchvision.datasets`, `torchaudio.datasets`, and `torchtext.datasets`

![meme 6](https://mat3e.github.io/brains/img/5.jpg)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32)

def mini_batch_gradient_descent(mod,X,y,learning_rate,num_epochs):
    opt = torch.optim.SGD(mod.parameters(), lr=learning_rate)
    
    for e in range(num_epochs):
        for x,y in loader:           # EASY
            opt.zero_grad()
            pred = mod(x)
            cost = torch.nn.functional.binary_cross_entropy(pred.squeeze(-1), y)

            cost.backward()     
            opt.step()                

learning_rate = 0.1
mini_batch_gradient_descent(neural_net,X,y,learning_rate,3)

# Level 7: And Beyond
See [here](https://pytorch.org/tutorials/beginner/nn_tutorial.html) for a summary or [here](https://pytorch.org/tutorials/beginner/basics/intro.html) for a full tutorial

![meme 7](https://mat3e.github.io/brains/img/6.jpg)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), 
                                transforms.Normalize((0.5,), (0.5,))])
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# Define model using Sequential
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

# Setup SGD optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Training loop
model.train()
for epoch in range(3):
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        # Using functional API for loss
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

print("Training complete!")

I got AI to generate some code to visualise the results of this model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get a batch of test data
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=16, shuffle=True)
images, labels = next(iter(test_loader))

# Make predictions
model.eval()
with torch.no_grad():
    outputs = model(images)
    predictions = outputs.argmax(dim=1)

# Create a 4x4 grid of images with predictions
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('Model Predictions on MNIST Test Set', fontsize=16)

for idx, ax in enumerate(axes.flat):
    # Unnormalize and convert to numpy
    img = images[idx].squeeze() * 0.5 + 0.5
    img = img.numpy()
    
    # Display image
    ax.imshow(img, cmap='gray')
    
    # Color code: green for correct, red for incorrect
    color = 'green' if predictions[idx] == labels[idx] else 'red'
    ax.set_title(f'Pred: {predictions[idx].item()}\nTrue: {labels[idx].item()}', 
                 color=color, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()